In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from PIL import Image
from dinosaw.models.vit_wrapper import AlibiVitWrapper, MODEL_LIST
from dinosaw.helpers import get_features
from dinosaw.utils import do_2D_pca
DEVICE="cuda:0"

flash attention installed


In [2]:
path_constant = "../../trained_models/alibi_coco_dv2_vits14_reg_ms.pth"
path_learned = "../../trained_models/alibi_coco_dv2_vits14_reg_ms_learned.pth"
# path_learned = "/home/pawlo/Arbeit/positional_bias/dino-saw/dinosaw/experiments/20260206_2352_alibi_wrap_leaned_slope_real_100_224_homog_bs16/best_model.pth"
HALF=False

In [3]:
alibi_learned = AlibiVitWrapper(
    model_identifier=MODEL_LIST[1],
    slope_type="learned",
    add_flash_attn=False,
    device=DEVICE
)
alibi_learned.load_state_dict(torch.load(path_learned, weights_only=True, map_location=DEVICE))


<All keys matched successfully>

In [4]:
alibi_const = AlibiVitWrapper(
    model_identifier=MODEL_LIST[1],
    slope_type="constant",
    add_flash_attn=False,
    device=DEVICE
)
alibi_const.load_state_dict(torch.load(path_constant, weights_only=True, map_location=DEVICE))

<All keys matched successfully>

In [5]:
models = [alibi_const, alibi_learned]
for model in models:
    model.eval()
    print(model.model.blocks[0].attn.m)
    if HALF:
        model.half()

tensor([[[1.]],

        [[1.]],

        [[1.]],

        [[1.]],

        [[1.]],

        [[1.]]], device='cuda:0')
Parameter containing:
tensor([[[0.9565]],

        [[0.8430]],

        [[1.4585]],

        [[0.2363]],

        [[0.1694]],

        [[1.2103]]], device='cuda:0', requires_grad=True)


In [7]:
SF=1
img_fname="wmg_si_c.png"
# img_fname="bimodal_zoom.png"
# img_fname="black_square.png"
# img_fname="NMC_2D_crop.png"

img_path = f"../../images/micro/{img_fname}"
img = Image.open(img_path).convert('RGB')
img = img.resize((int(SF * img.width), int(SF * img.height)), Image.LANCZOS)


In [8]:
feats_reduced = {channel_group: {} for channel_group in range(3)}

for i, model in enumerate(models):
    feats = get_features(model, img, to_half=HALF, device=DEVICE)
    for channel_group in range(3):
        feats_reduced[channel_group][i] = do_2D_pca(feats, (channel_group+1)*3, pre_norm="std", post_norm='minmax')[:, :, channel_group*3:channel_group*3+3]

In [23]:
plt.style.use("thesis.mplstyle")

W,H = 7,2.3
NROWS, NCOLS = 2,4
fig = plt.figure(figsize=(W,H))
gs = GridSpec(NROWS, NCOLS, figure=fig)

img_ax = fig.add_subplot(gs[:, 0])
img_ax.imshow(img)
img_ax.set_yticks([])
img_ax.set_xticks([])
img_ax.set_title("SiGr anode SEM")

for j, _ in enumerate(models):
    for i, channel_group in enumerate(list(range(3))):
        ax = fig.add_subplot(gs[j, i + 1])
        ax.imshow(feats_reduced[channel_group][j])
        ax.set_yticks([])
        ax.set_xticks([])
        if j==0:
            ax.set_title(f"PCs {1 + channel_group*3} to {1 + channel_group*3+2}")
        if i==0:
            if j==0:
                ax.set_ylabel(r"Constant $m$",  weight="bold")
            else:
                ax.set_ylabel(r"Learned $m$")

# plt.tight_layout()
plt.savefig("out/const_vs_learned_m.pdf", dpi=300)
plt.close()